In [1]:
import time
import torch
import numpy as np
from pathlib import Path

# LeRobot imports
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.robots.so100_follower.config_so100_follower import SO100FollowerConfig
from lerobot.robots.so100_follower.so100_follower import SO100Follower
from lerobot.utils.control_utils import init_keyboard_listener, predict_action
from lerobot.utils.utils import (log_say, get_safe_torch_device)


In [2]:
INFERENCE_TIME_SEC = 30  # How long to run inference
FPS = 30                 # Control frequency
DEVICE = "mps"          # "cuda", "mps", or "cpu"
TASK_DESCRIPTION = "Put the red block in the blue bin"

ROBOT_PORT = "/dev/tty.usbmodem58760435551"  # Update for your robot
CAMERA_CONFIG = {
    "front": OpenCVCameraConfig(index_or_path=2, width=1920, height=1080, fps=FPS),
    "top": OpenCVCameraConfig(index_or_path=0, width=1920, height=1080, fps=FPS)
}

POLICY_TYPE = "smolvla"  # "act" or "smolvla"
#POLICY_PATH = "/Users/shreyas/Downloads/robots/policies/smolvla/020000/pretrained_model"
POLICY_PATH = "/Users/shreyas/Downloads/robots/policies/tedrake/020000/pretrained_model"

In [13]:
# Create robot configuration
robot_config = SO100FollowerConfig(
    port=ROBOT_PORT, 
    id="red_arm", 
    cameras=CAMERA_CONFIG
)

# Initialize robot
robot = SO100Follower(robot_config)
print(f"✓ Robot configured on port: {ROBOT_PORT}")


✓ Robot configured on port: /dev/tty.usbmodem58760435551


In [14]:
# Load policy (automatically handles Tedrake normalization if trained with it)
if POLICY_TYPE == "act":
    policy = ACTPolicy.from_pretrained(POLICY_PATH)
elif POLICY_TYPE == "smolvla":
    policy = SmolVLAPolicy.from_pretrained(POLICY_PATH)
else:
    raise ValueError(f"Unsupported policy type: {POLICY_TYPE}")

policy.to(DEVICE)
print(f"✓ Loaded {POLICY_TYPE} policy from: {POLICY_PATH}")
print(f"✓ Policy moved to device: {DEVICE}")


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...
Loading weights from local directory
✓ Loaded smolvla policy from: /Users/shreyas/Downloads/robots/policies/tedrake/020000/pretrained_model
✓ Policy moved to device: mps


In [15]:
# Setup keyboard listener for real-time control
#_, events = init_keyboard_listener()
#print("✓ Keyboard listener initialized")
#print("   ESC: Stop | SPACE: Pause/Resume | R: Reset")


In [16]:
from lerobot.datasets.utils import build_dataset_frame, hw_to_dataset_features
# Configure the dataset features
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

In [21]:
# Connect to robot
robot.connect()
print("✓ Robot connected successfully")

# Utility function for precise timing
def busy_wait(duration):
    """Precise timing for control loop"""
    if duration > 0:
        end_time = time.perf_counter() + duration
        while time.perf_counter() < end_time:
            pass

print(f"Starting inference for {INFERENCE_TIME_SEC} seconds...")
print(f"Task: {TASK_DESCRIPTION}")

✓ Robot connected successfully
Starting inference for 30 seconds...
Task: Put the red block in the blue bin


In [22]:
start_time = time.perf_counter()
num_steps = int(INFERENCE_TIME_SEC * FPS)
paused = False

try:
    for step in range(num_steps):
        loop_start = time.perf_counter()
        
        # Capture observation from robot
        observation = robot.get_observation()
        
        observation_frame = build_dataset_frame(dataset_features, observation, prefix="observation")
        action_values = predict_action(
                observation_frame,
                policy,
                get_safe_torch_device(policy.config.device),
                policy.config.use_amp,
                task=TASK_DESCRIPTION,
                robot_type=robot.robot_type,
            )
        action = {key: action_values[i].item() for i, key in enumerate(robot.action_features)}
        
        # Send action to robot
        robot.send_action(action)
        
        # Progress logging
        if step % (FPS * 5) == 0:  # Every 5 seconds
            elapsed = time.perf_counter() - start_time
            remaining = INFERENCE_TIME_SEC - elapsed
            print(f"Step {step}/{num_steps} | {elapsed:.1f}s elapsed | {remaining:.1f}s remaining")
        
        # Maintain precise timing
        dt = time.perf_counter() - loop_start
        busy_wait(1/FPS - dt)
        
except KeyboardInterrupt:
    print("\n Inference interrupted by user")
    
except Exception as e:
    print(f"\n Error during inference: {e}")
    
finally:
    # Always disconnect robot
    robot.disconnect()
    print("\n✓ Robot disconnected safely")
    
total_time = time.perf_counter() - start_time
print(f" Inference completed in {total_time:.1f} seconds")
print(f" Average FPS: {step / total_time:.1f}")


Step 0/900 | 0.9s elapsed | 29.1s remaining
Step 150/900 | 8.2s elapsed | 21.8s remaining
Step 300/900 | 15.5s elapsed | 14.5s remaining
Step 450/900 | 22.8s elapsed | 7.2s remaining
Step 600/900 | 30.1s elapsed | -0.1s remaining
Step 750/900 | 37.5s elapsed | -7.5s remaining

✓ Robot disconnected safely
 Inference completed in 44.2 seconds
 Average FPS: 20.3


In [ ]:
# Example: Add custom state information (like in robotbuilder inference)
def add_custom_state(observation, pick_target=None, place_target=None):
    """Add custom state information to observation"""
    if pick_target is not None and place_target is not None:
        # Normalize targets to [0,1] if needed
        norm_pick = torch.tensor(pick_target, dtype=torch.float32)
        norm_place = torch.tensor(place_target, dtype=torch.float32)
        
        # Append to state
        observation["observation.state"] = torch.cat([
            observation["observation.state"], 
            norm_pick, 
            norm_place
        ])
    
    return observation

# Usage in inference loop:
# observation = add_custom_state(observation, pick_target=[0.5, 0.3], place_target=[0.8, 0.7])
